In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Veriyi yükle ve eksik satırları sil
df = pd.read_excel("ayakkabi_numarasi.xlsx")
df = df.dropna()

df.head()

,Kilo(kg),Boy(metre),Yaş,Cinsiyet(E veya K),Ayakkabı Numarası
0,85,1.74,42,E,41.0
1,44,1.40,10,E,36.0
2,32,1.38,11,E,36.0
3,48,1.46,11,E,37.0
4,75,1.62,47,E,42.0


In [7]:
# 2. Boy sütunundaki birim karmaşasını çözme
# Eğer boy 3'ten büyükse (örneğin 140.00), bunun santimetre olduğunu varsayıp 100'e bölerek metreye çeviriyoruz.
df['Boy(metre)'] = df['Boy(metre)'].apply(lambda x: x / 100 if x > 3 else x)

# 3. Kategorik veriyi sayısallaştırma (Encoding)
# 'E' (Erkek) için 1, 'K' (Kadın) için 0 atayalım.
df['Cinsiyet(E veya K)'] = df['Cinsiyet(E veya K)'].map({'E': 1, 'K': 0})

df.tail()

,Kilo(kg),Boy(metre),Yaş,Cinsiyet(E veya K),Ayakkabı Numarası
55,40,1.53,10,1,37.0
56,40,1.54,11,1,38.0
57,39,1.52,11,0,34.0
58,40,1.48,11,1,38.0
59,39,1.49,10,0,34.0


In [8]:
df["Cinsiyet(E veya K)"].value_counts()

Cinsiyet(E veya K)
1    31
0    27
Name: count, dtype: int64

In [9]:
# 4. Özellikler (X) ve Hedef (y) olarak ayırma
# Ayakkabı numarası tahmin edeceğimiz hedef, geri kalanlar ise girdilerimizdir.
X = df[['Kilo(kg)', 'Boy(metre)', 'Yaş', 'Cinsiyet(E veya K)']].values
y = df[['Ayakkabı Numarası']].values

In [11]:
print(type(X))
print(type(y))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [12]:
# 5. Verileri Ölçeklendirme (Standardization)
# Verilerin ortalamasını 0, standart sapmasını 1 yapacak şekilde ölçeklendirir.
scaler = StandardScaler()
X_olcekli = scaler.fit_transform(X)

# 6. Veriyi Eğitim (%80) ve Test (%20) olarak ayırma
X_train, X_test, y_train, y_test = train_test_split(X_olcekli, y, test_size=0.2, random_state=42)

# 7. Numpy dizilerini PyTorch Tensor formatına çevirme
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [15]:
# 8. Modeli Tanımlama (Regresyon Modeli)
class AyakkabiTahminModeli(nn.Module):
    def __init__(self):
        super(AyakkabiTahminModeli, self).__init__()
        # Giriş: 4 Özellik -> Gizli Katmanlar -> Çıkış: 1 Tahmin
        self.ag_yapisi = nn.Sequential(
            nn.Linear(4, 16),  # 4 giriş, 16 nöronlu ilk gizli katman
            nn.ReLU(),         # Negatifleri sıfırla
            nn.Linear(16, 8),  # 16 giriş, 8 nöronlu ikinci gizli katman
            nn.ReLU(),
            nn.Linear(8, 1)    # 8 giriş, tek bir ayakkabı numarası çıkışı
        )

    def forward(self, x):
        return self.ag_yapisi(x)

# Modeli oluştur
model = AyakkabiTahminModeli()

In [16]:
#modelin parametrelerini ve yapısını görüntüle
print(model)

AyakkabiTahminModeli(
  (ag_yapisi): Sequential(
    (0): Linear(in_features=4, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=1, bias=True)
  )
)


In [19]:
# 9. Kayıp Fonksiyonu ve Optimizasyon
criterion = nn.MSELoss()  # Regresyon için Ortalama Karesel Hata
optimizer = optim.Adam(model.parameters(), lr=0.01)  # Öğrenme katsayısı

# 10. Eğitim Döngüsü (Training Loop)
epoch_sayisi = 500

for epoch in range(epoch_sayisi):
    model.train() # Modeli eğitim moduna al
    
    # İleri yayılım (Tahmin)
    tahminler = model(X_train_tensor)
    
    # Hatayı hesapla
    loss = criterion(tahminler, y_train_tensor)
    
    # Geri yayılım ve ağırlıkların güncellenmesi
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Her 100 adımda bir süreci ekrana yazdır
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epoch_sayisi}], Hata (MSE Loss): {loss.item():.4f}")

Epoch [100/500], Hata (MSE Loss): 1.4893
Epoch [200/500], Hata (MSE Loss): 0.6419
Epoch [300/500], Hata (MSE Loss): 0.5493
Epoch [400/500], Hata (MSE Loss): 0.4885
Epoch [500/500], Hata (MSE Loss): 0.4460


In [20]:
# 11. Test Aşaması
model.eval() # Modeli test moduna al (dropout vs. varsa kapatır)

with torch.no_grad(): # Test sırasında ağırlıkları güncellemeyi durdur
    test_tahminleri = model(X_test_tensor)
    test_loss = criterion(test_tahminleri, y_test_tensor)
    print(f"\nTest Verisi Üzerindeki Ortalama Karesel Hata: {test_loss.item():.4f}")
    
    # Gerçek değerler ile tahminleri karşılaştırmak için yan yana görelim
    print("\nİlk 5 Test Verisi İçin Karşılaştırma:")
    for i in range(min(5, len(y_test_tensor))):
        gercek = y_test_tensor[i].item()
        tahmin = test_tahminleri[i].item()
        # Tahmin edilen numara küsuratlı çıkacaktır, yuvarlamak mantıklıdır
        print(f"Gerçek Numara: {gercek} | Modelin Tahmini: {tahmin:.1f} (Yuvarlanmış: {round(tahmin)})")


Test Verisi Üzerindeki Ortalama Karesel Hata: 1.8373

İlk 5 Test Verisi İçin Karşılaştırma:
Gerçek Numara: 41.0 | Modelin Tahmini: 43.8 (Yuvarlanmış: 44)
Gerçek Numara: 37.0 | Modelin Tahmini: 37.1 (Yuvarlanmış: 37)
Gerçek Numara: 38.0 | Modelin Tahmini: 36.8 (Yuvarlanmış: 37)
Gerçek Numara: 37.0 | Modelin Tahmini: 38.5 (Yuvarlanmış: 38)
Gerçek Numara: 41.0 | Modelin Tahmini: 40.5 (Yuvarlanmış: 40)


In [21]:
import torch
import joblib

# Öğrenilmiş ağırlıkları (weights & biases) .pth uzantısıyla kaydet
torch.save(model.state_dict(), "ayakkabi_modeli.pth")

# Eğitimde kullanılan scaler'ı kaydet (yeni gelen veriyi aynı oranda küçültmek için zorunludur)
joblib.dump(scaler, "veri_olceklendirici.pkl")

print("Model ve Scaler başarıyla kaydedildi.")
# Modeli TorchScript formatına çevir (JIT Compiler)
script_model = torch.jit.script(model)

# Tek parça halinde kaydet
script_model.save("bagimsiz_model.pt")

Model ve Scaler başarıyla kaydedildi.


/home/ogretmen/Belgeler/2026_2027/oyg2a1/.venv/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
